In [1]:
!pip install opensmile librosa torchaudio transformers scikit-learn tqdm

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 706.0/706.0 kB 1.8 MB/s eta 0:00:001.8 MB/s eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 5.7 MB/s eta 0:00:00 MB/s eta 0:00:01:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 70.3/70.3 kB 9.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 3.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.9/41.9 kB 5.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 150.9/150.9 kB 8.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 10.0 MB/s eta 0:00:00m eta 0:00:010:01:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.2/84.2 kB 7.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 324.8/324.8 kB 6.8 MB/s eta 0:00:000:00:01


In [4]:
import os
import numpy as np
from tqdm import tqdm
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
import opensmile

# Feature extraction with OpenSMILE
def extract_opensmile_features(dataset_path):
    smile = opensmile.Smile(
        feature_set=opensmile.FeatureSet.ComParE_2016,
        feature_level=opensmile.FeatureLevel.Functionals,
    )
    features = []
    labels = []

    for label, folder in enumerate(['real', 'fake']):
        folder_path = os.path.join(dataset_path, folder)
        files = [f for f in os.listdir(folder_path) if f.endswith('.wav')]
        print(f"Extracting features from '{folder}' files...")
        for file in tqdm(files, desc=f"Processing {folder}", unit="file"):
            file_path = os.path.join(folder_path, file)
            feat = smile.process_file(file_path)
            features.append(feat.values.flatten())
            labels.append(label)
    
    return np.array(features), np.array(labels)

# Main workflow
dataset_path = '/Users/aviralchauhan/College/Sem8/hcai/final_project/deepfake_detection/dataset'
X, y = extract_opensmile_features(dataset_path)

# Train-test split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Train model
clf = RandomForestClassifier(n_estimators=100, random_state=42)
clf.fit(X_train, y_train)

# Evaluate
y_pred = clf.predict(X_test)
print(f"Accuracy: {accuracy_score(y_test, y_pred):.4f}")
print(f"Precision: {precision_score(y_test, y_pred):.4f}")
print(f"Recall: {recall_score(y_test, y_pred):.4f}")
print(f"F1-score: {f1_score(y_test, y_pred):.4f}")

Extracting features from 'real' files...


Processing real: 100%|██████████| 2525/2525 [05:58<00:00,  7.04file/s]


Extracting features from 'fake' files...


Processing fake: 100%|██████████| 10660/10660 [18:38<00:00,  9.53file/s] 


Accuracy: 0.9992
Precision: 0.9991
Recall: 1.0000
F1-score: 0.9995


In [5]:
import joblib

# Save the trained model
model_path = 'models/random_forest_opensmile_model.pkl'
joblib.dump(clf, model_path)
print(f"Model saved to {model_path}")

Model saved to models/random_forest_opensmile_model.pkl
